# Simple CNN Image Classification

## Learning goals
- Understand why image data needs a different modeling approach than tabular data.
- Build a small CNN in PyTorch.
- Train on FashionMNIST with a Colab-friendly setup.
- Read training curves and a confusion matrix.
- Inspect correct and incorrect predictions.


In [ ]:
# Notebook-specific environment setup.
# We install missing packages so this notebook can be opened directly in Colab.

import importlib.util
import subprocess
import sys


REQUIRED_PACKAGES = {
    "torch": "torch",
    "torchvision": "torchvision",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
}

for import_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"{import_name} is already installed.")


## Why images are different

In tabular data, each column is a separate feature. In image data:
- each example is a grid of pixels
- nearby pixels matter together
- patterns like edges and shapes can appear in different positions

**Why CNNs help**

Convolutional layers look for local patterns and reuse those learned filters across the image.
That makes CNNs much better suited to image tasks than simple models on raw pixels.


In [ ]:
# Import the libraries needed for deep learning, evaluation, and plotting.
# We keep the setup explicit so learners can see the main pieces of a PyTorch workflow.

import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms


# Set all relevant random seeds for more reproducible behavior.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Choose GPU if available because training is much faster there.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Class names provided by FashionMNIST.
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

# Keep the default run light enough for a workshop.
BATCH_SIZE = 128 if torch.cuda.is_available() else 64
EPOCHS = 4 if torch.cuda.is_available() else 3
USE_SMALL_SUBSET = True
MAX_TRAIN_SAMPLES = 20000 if torch.cuda.is_available() else 8000
MAX_VAL_SAMPLES = 4000 if torch.cuda.is_available() else 2000
MAX_TEST_SAMPLES = 4000 if torch.cuda.is_available() else 2000

sns.set_theme(style="whitegrid")


## Important lines in the setup cell

- `torch.device("cuda" if ... else "cpu")`: switches automatically between GPU and CPU.
- `torch.manual_seed(SEED)`: makes training behavior more repeatable.
- `USE_SMALL_SUBSET = True`: keeps runtime short for workshops and CPU users.
- `class_names = [...]`: maps numeric labels like `0` or `1` to readable category names.


In [ ]:
# Convert each image into a tensor and normalize pixel values.
# FashionMNIST images are grayscale, so each image has 1 channel.

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ]
)

# Download the training split and the official test split.
full_train_dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

full_test_dataset = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

# Split the official training data into train and validation subsets.
train_size = 50000
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)


def take_subset(dataset, max_samples, seed=SEED):
    # This helper keeps only a random subset when we want faster experiments.
    if max_samples >= len(dataset):
        return dataset

    rng = np.random.default_rng(seed)
    chosen_indices = rng.choice(len(dataset), size=max_samples, replace=False)
    return Subset(dataset, chosen_indices.tolist())


if USE_SMALL_SUBSET:
    train_dataset = take_subset(train_dataset, MAX_TRAIN_SAMPLES, seed=SEED)
    val_dataset = take_subset(val_dataset, MAX_VAL_SAMPLES, seed=SEED + 1)
    test_dataset = take_subset(full_test_dataset, MAX_TEST_SAMPLES, seed=SEED + 2)
else:
    test_dataset = full_test_dataset

print(f"Train examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")
print(f"Test examples: {len(test_dataset)}")


## What the data loading lines do

- `transforms.ToTensor()`: converts each image into a PyTorch tensor.
- `transforms.Normalize((0.5,), (0.5,))`: rescales pixel values to a more model-friendly range.
- `datasets.FashionMNIST(...)`: downloads and loads the dataset.
- `random_split(...)`: creates separate train and validation subsets.
- `take_subset(...)`: reduces dataset size for speed while keeping the demo meaningful.


In [ ]:
# Show a few example images before training.
# Looking at the data is a good habit in every ML workflow.

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for ax, index in zip(axes, range(10)):
    image, label = train_dataset[index]

    # Undo normalization so the image looks natural in the plot.
    image_for_plot = image.squeeze() * 0.5 + 0.5

    ax.imshow(image_for_plot, cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")

plt.suptitle("Sample FashionMNIST images", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Create data loaders that feed mini-batches into the model.
# Batch tensors for images usually look like:
# [batch_size, channels, height, width]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

sample_images, sample_labels = next(iter(train_loader))

print(f"Image batch shape: {sample_images.shape}")
print(f"Label batch shape: {sample_labels.shape}")
print("Interpretation: [batch_size, channels, height, width]")


## A small CNN

Our network has these parts:
- **convolution** to detect local patterns
- **ReLU activation** to add non-linearity
- **max pooling** to shrink spatial size
- **classifier head** to map learned features to class scores


In [ ]:
# Define a compact CNN that is fast enough for a workshop.
# The comments show how tensor shapes change through the network.

class SimpleCNN(nn.Module):
    def __init__(
        self,
        conv1_channels=16,
        conv2_channels=32,
        kernel_size=3,
        dropout_rate=0.2,
        num_classes=10,
    ):
        super().__init__()

        # Padding keeps the height and width stable before pooling.
        padding = kernel_size // 2

        self.features = nn.Sequential(
            # Input shape: [batch, 1, 28, 28]
            nn.Conv2d(1, conv1_channels, kernel_size=kernel_size, padding=padding),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # -> [batch, conv1_channels, 14, 14]
            nn.Conv2d(
                conv1_channels,
                conv2_channels,
                kernel_size=kernel_size,
                padding=padding,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # -> [batch, conv2_channels, 7, 7]
        )

        # Use a dummy tensor so the flattened size is computed automatically.
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 28, 28)
            flattened_size = self.features(dummy).view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(flattened_size, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # Pass the image through the convolutional feature extractor.
        x = self.features(x)

        # Pass the extracted features through the classifier head.
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
print(model)


## Important CNN lines

- `nn.Conv2d(...)`: learns small image filters.
- `nn.MaxPool2d(kernel_size=2)`: halves the image height and width after each pooling layer.
- `nn.Flatten()`: turns the feature map grid into one long vector for the classifier head.
- `nn.Dropout(...)`: randomly drops some activations during training to reduce overfitting.
- `model = SimpleCNN().to(device)`: moves the model to GPU if one is available.


In [ ]:
# Set up the loss function and optimizer.
# CrossEntropyLoss is standard for multi-class classification.

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def train_one_epoch(model, loader, optimizer, criterion, device):
    # model.train() turns on training behavior such as dropout.
    model.train()

    running_loss = 0.0
    total_examples = 0

    for images, labels in loader:
        # Move the current batch to CPU or GPU.
        images = images.to(device)
        labels = labels.to(device)

        # Clear old gradients from the previous batch.
        optimizer.zero_grad()

        # Forward pass: compute class scores for this batch.
        logits = model(images)

        # Compute the loss by comparing predicted scores with true labels.
        loss = criterion(logits, labels)

        # Backward pass: compute gradients for all trainable parameters.
        loss.backward()

        # Optimizer step: update model weights using the gradients.
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total_examples += images.size(0)

    return running_loss / total_examples


def evaluate(model, loader, criterion, device):
    # model.eval() turns off training-only behavior such as dropout.
    model.eval()

    running_loss = 0.0
    total_examples = 0
    all_predictions = []
    all_labels = []

    # torch.no_grad() saves memory and compute during evaluation.
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)
            predictions = logits.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            total_examples += images.size(0)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    average_loss = running_loss / total_examples
    accuracy = accuracy_score(all_labels, all_predictions)

    return average_loss, accuracy, np.array(all_labels), np.array(all_predictions)


## What the training loop lines do

- `optimizer.zero_grad()`: resets old gradients before the new batch.
- `logits = model(images)`: runs the forward pass.
- `loss = criterion(logits, labels)`: measures how wrong the predictions are.
- `loss.backward()`: computes gradients.
- `optimizer.step()`: updates the model weights.
- `model.eval()` and `torch.no_grad()`: make evaluation faster and more stable.


In [ ]:
# Train for a few epochs and store the history so we can plot it later.

history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
}

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_accuracy, _, _ = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val accuracy: {val_accuracy:.4f}"
    )

elapsed = time.time() - start_time
print(f"Training finished in {elapsed / 60:.2f} minutes.")


In [ ]:
# Plot the training curves.
# These curves help us see if the model is learning and whether validation starts to flatten out.

epochs = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], marker="o", label="Train loss")
axes[0].plot(epochs, history["val_loss"], marker="o", label="Validation loss")
axes[0].set_title("Loss by epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs, history["val_accuracy"], marker="o", color="green")
axes[1].set_title("Validation accuracy by epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")

plt.tight_layout()
plt.show()


## Test evaluation

We used the validation set while training and choosing settings.
Now we evaluate once on the held-out test set.


In [ ]:
# Evaluate on the held-out test set and inspect class-level behavior.

test_loss, test_accuracy, test_labels, test_predictions = evaluate(
    model, test_loader, criterion, device
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_predictions,
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0,
    )
)

cm = confusion_matrix(
    test_labels,
    test_predictions,
    labels=list(range(len(class_names))),
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title("Confusion matrix on the test set")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Collect a few correct and incorrect predictions.
# Looking at errors builds intuition much faster than a single accuracy number.

model.eval()
correct_examples = []
incorrect_examples = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        predictions = logits.argmax(dim=1)

        for image, true_label, predicted_label in zip(
            images.cpu(), labels.cpu(), predictions.cpu()
        ):
            example = (image.squeeze(), int(true_label), int(predicted_label))

            if true_label == predicted_label and len(correct_examples) < 6:
                correct_examples.append(example)
            elif true_label != predicted_label and len(incorrect_examples) < 6:
                incorrect_examples.append(example)

        if len(correct_examples) >= 6 and len(incorrect_examples) >= 6:
            break


def show_examples(examples, title):
    fig, axes = plt.subplots(1, len(examples), figsize=(14, 3))

    for ax, (image, true_label, predicted_label) in zip(axes, examples):
        image_for_plot = image * 0.5 + 0.5
        ax.imshow(image_for_plot, cmap="gray")
        ax.set_title(
            f"True: {class_names[true_label]}\nPred: {class_names[predicted_label]}",
            fontsize=9,
        )
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


show_examples(correct_examples, "A few correct predictions")
show_examples(incorrect_examples, "A few incorrect predictions")


In [ ]:
# Playground: choose a few test examples by index and inspect the model's predictions.
# This gives learners a simple way to experiment without retraining.

playground_indices = [0, 1, 2, 3, 4]

fig, axes = plt.subplots(1, len(playground_indices), figsize=(15, 3))

for ax, idx in zip(axes, playground_indices):
    image, true_label = test_dataset[idx]

    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(device))
        predicted_label = int(logits.argmax(dim=1).item())

    image_for_plot = image.squeeze() * 0.5 + 0.5
    ax.imshow(image_for_plot, cmap="gray")
    ax.set_title(
        f"Idx {idx}\nTrue: {class_names[true_label]}\nPred: {class_names[predicted_label]}",
        fontsize=9,
    )
    ax.axis("off")

plt.suptitle("Playground predictions on chosen test indices")
plt.tight_layout()
plt.show()


## Think about it

- Why are CNNs a better fit here than logistic regression on raw pixels?
- Which clothing classes look easiest? Which look hardest?
- If this were a real product, what would you evaluate besides accuracy?
- When might a smaller or local model be enough?


## Exercise 1: change the architecture

In the next cell, change one of these:
- `conv1_channels`
- `conv2_channels`
- `kernel_size`
- `dropout_rate`

Then train for one short run and compare the validation accuracy.


In [ ]:
# Try a slightly different architecture.
# This is a fast experiment rather than a full model search.

exercise_model = SimpleCNN(
    conv1_channels=32,
    conv2_channels=64,
    kernel_size=3,
    dropout_rate=0.3,
).to(device)

exercise_optimizer = torch.optim.Adam(exercise_model.parameters(), lr=1e-3)
exercise_epochs = 1

for epoch in range(1, exercise_epochs + 1):
    exercise_train_loss = train_one_epoch(
        exercise_model, train_loader, exercise_optimizer, criterion, device
    )
    exercise_val_loss, exercise_val_accuracy, _, _ = evaluate(
        exercise_model, val_loader, criterion, device
    )

    print(
        f"Exercise epoch {epoch}/{exercise_epochs} | "
        f"Train loss: {exercise_train_loss:.4f} | "
        f"Val loss: {exercise_val_loss:.4f} | "
        f"Val accuracy: {exercise_val_accuracy:.4f}"
    )


## Exercise 2: experiment with runtime choices

Try one or more of these:
- set `USE_SMALL_SUBSET = False`
- train for fewer or more epochs
- change `playground_indices`
- inspect the wrong predictions and discuss why the model failed

**Recap**

In this notebook, you built and trained a small CNN from scratch, read training curves, evaluated on a test set, and inspected both successes and failures.
